In [1]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
#importer les packages
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
#saisir l'Url
url =

SyntaxError: invalid syntax (1398283569.py, line 2)

In [4]:


def get_country_links(base_url):
    response = requests.get(base_url)
    links = {}
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        footer = soup.find("footer", id="footer")
        if footer:
            accordians = footer.find_all("div", class_="accordian")
            for bloc in accordians:
                title = bloc.find("div", class_="accordian_title")
                if title and "Pays" in title.text.strip():
                    for a in bloc.find_all("a"):
                        country_name = a.text.strip()
                        country_url = a.get("href")
                        if not country_url.startswith("http"):
                            country_url = "https://www.dabadoc.com" + country_url
                        links[country_name] = country_url
    return links


def get_frequent_searches(url):
    response = requests.get(url)
    results = []
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        footer = soup.find("footer", id="footer")
        if footer:
            accordians = footer.find_all("div", class_="accordian")
            for bloc in accordians:
                title = bloc.find("div", class_="accordian_title")
                if title and "Recherches fréquentes" in title.text.strip():
                    for a in bloc.find_all("a"):
                        search = a.text.strip()
                        search_url = a.get("href")
                        if not search_url.startswith("http"):
                            search_url = "https://www.dabadoc.com" + search_url
                        results.append((search, search_url))
    return results


def extract_doctors(search_url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(search_url, headers=headers)
    data = []

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        doctors = soup.find_all("div", class_="result-box")

        for doc in doctors:
            name_tag = doc.find("h2")
            speciality_tag = doc.find("p")
            link_tag = name_tag.find("a") if name_tag else None

            if name_tag and speciality_tag and link_tag:
                name = name_tag.get_text(strip=True)
                speciality = speciality_tag.get_text(strip=True)
                link = link_tag["href"]
                full_link = link if link.startswith("http") else "https://www.dabadoc.com" + link

                
                detail_resp = requests.get(full_link, headers=headers)
                if detail_resp.status_code == 200:
                    detail_soup = BeautifulSoup(detail_resp.text, 'html.parser')
                    
                    
                    cards = detail_soup.find_all("div", class_="card")
                    access = ""
                    for c in cards:
                        title = c.find("h3")
                        if title and "Indications" in title.get_text():
                            card_body = c.find("div", class_="card-text")
                            if card_body:
                                access = card_body.get_text(strip=True)
                            break

                    data.append({
                        "Nom": name,
                        "Spécialité": speciality,
                        "Lien": full_link,
                        "Accès": access
                    })
    return data


pays_liens = get_country_links("https://www.dabadoc.com/tn")
lien_recherches = []
for nom_pays, url_pays in pays_liens.items():
    recherches = get_frequent_searches(url_pays)
    for label, lien in recherches:
        lien_recherches.append((label, lien))



df_total = pd.DataFrame()
for label, lien in lien_recherches[0:2]:
    try:
        doctors_data = extract_doctors(lien)
        df = pd.DataFrame(doctors_data)
        df_total = pd.concat([df_total, df], ignore_index=True)
    except Exception as e:
        print(f"Erreur avec {label} : {e}")


df_total.to_csv("doctors_data.csv", index=False)
print("Export terminé : doctors_data.csv")


Export terminé : doctors_data.csv


In [5]:
df_total.head()

,Nom,Spécialité,Lien,Accès
0,Dr Faissel Bennouna,"Dentiste, Endodontiste, Esthétique dentaire, I...",https://www.dabadoc.com/ma/dentiste/casablanca...,Marina Center Angle Bd Zerktouni Et Bd De La C...
1,Dr Mouhssine Alj,Dentiste à Casablanca,https://www.dabadoc.com/ma/dentiste/casablanca...,"12 Rua Ras Al Maa, Villa, Hay Hassani, ALJ Exc..."
2,Dr Akesbi Jihane,"Dentiste, Endodontiste, Orthodontiste, Parodon...",https://www.dabadoc.com/ma/dentiste/casablanca...,"9 Rue Abou Maâchar,, Résidence Abou Maâchar, 2..."
3,Dr Sara Barkaoui,"Chirurgie buccale, Dentiste, Endodontiste, Est...",https://www.dabadoc.com/ma/chirurgie-buccale/c...,"28, Rue Hafid Ibrahim( Ex Chateaubriand) 1er é..."
4,Dr Zineb El Menjra,"Dentiste, Esthétique dentaire, Endodontiste, O...",https://www.dabadoc.com/ma/dentiste/casablanca...,"26 Angle Omar Slaoui Et Rue D'agadir ,3 ème ét..."
